In [1]:
# TODO 1: 
# Cargar las plantillas y los recursos con pandas
import pandas as pd
print(pd.__version__)

3.0.3


In [2]:
data = r'C:\Users\P5E002~1\AppData\Local\Temp\trabajo_git\IDSD422-lab3-diseno-experimento\plantillas\registro_experimento.csv'
df = pd.read_csv(data)

In [15]:
# TODO 2: calcula e imprime el promedio de ventas_producto por ubicacion 
promedio = df.groupby('ubicacion')['ventas_producto'].mean()
promedio

ubicacion
habitual             1040.0
pasillo_principal    1335.0
Name: ventas_producto, dtype: float64

In [17]:
# TODO 3: imprime el número de registros por grupo (value_counts)\

print(df['ubicacion'].value_counts())

ubicacion
pasillo_principal    2
habitual             2
Name: count, dtype: int64


In [23]:
# Diseño de experimento
import numpy as np
tiendas = ["Quito Norte", "Quito Sur", "Guayaquil Centro", "Guayaquil Norte", 
           "Cuenca Centro", "Cuenca Sur", "Ambato Centro", "Manta", "Loja", "Machala"]

semanas = [1, 2, 3, 4] 

rng = np.random.default_rng(1234597)
primera = rng.choice(["pasillo_principal", "habitual"], size=len(tiendas))
print(primera)

['habitual' 'pasillo_principal' 'habitual' 'pasillo_principal'
 'pasillo_principal' 'habitual' 'habitual' 'habitual' 'habitual'
 'pasillo_principal']


In [26]:
filas = []

for tienda, cond in zip(tiendas, primera):
    if cond == 'pasillo_principal':
        otra = 'habitual'
    else:
        otra = 'pasillo_principal'
    for s in semanas:
        filas.append({"tienda": tienda, 
                       "semana": s,
                       "ubicacion": cond if s%2 == 1 else otra})
diseno = pd.DataFrame(filas)

In [27]:
rng = np.random.default_rng(1234598)  # Semilla para reproducibilidad
efecto_tienda = dict(zip(tiendas, rng.normal(1100, 120, len(tiendas)))) 
efecto_semana = {1: 0, 2: 30, 3: 60, 4: 90}  
diseno["ventas_producto"] = (
     diseno["tienda"].map(efecto_tienda) +
     diseno["semana"].map(efecto_semana) + 
     (diseno["ubicacion"] == "pasillo_principal")*220 +
     rng.normal(0, 90, len(diseno))
 ).round(0).astype(int)

In [29]:
#Promocion simultanea registrada
diseno["promocion_simultanea"] = "no"

diseno.loc[(diseno["tienda"]== "Guayaquil Centro") & (diseno["semana"] == 1), "promocion_simultanea"] = "si"
diseno.loc[diseno["promocion_simultanea"] == "si", "ventas_producto"] += 180

diseno.to_csv(r"C:\Users\P5E002~1\AppData\Local\Temp\trabajo_git\IDSD422-lab3-diseno-experimento\plantillas/resultado_simulacion.csv", index=False)

#Analisis pareado: una fila por tienda con sus condiciones 
piv = diseno.groupby(["tienda", "ubicacion"])["ventas_producto"].mean().unstack()
piv["diferencia"] = piv["pasillo_principal"] - piv["habitual"]

media = piv["diferencia"].mean()
dev = piv["diferencia"].std(ddof=1)

print("\n")
print("Promedio de la diferencia de ventas entre ubicaciones:", media)
print("Desviación estándar de la diferencia de ventas entre ubicaciones:", dev)




Promedio de la diferencia de ventas entre ubicaciones: 230.7
Desviación estándar de la diferencia de ventas entre ubicaciones: 105.11269085023834


In [1]:
import scipy.stats as stats

n = len(piv)

# Se importó scipy.stats como stats para corregir el NameError
t_crict = stats.t.ppf(0.975, df = 9)
ic_inf = media - (t_crict*dev/n**0.5)
print("Límite inferior del intervalo de confianza:", ic_inf)
ic_sup = media + (t_crict*dev/n**0.5)
print("Límite superior del intervalo de confianza:", ic_sup)

print(f"[{ic_inf} , {ic_sup}]")

#Prueba p-value
prueba = stats.ttest_rel(piv["pasillo_principal"], piv["habitual"])
print(f"{prueba.pvalue: .10f}")

NameError: name 'piv' is not defined